# Training Dataset Catalog

Source-of-truth reference for **public training datasets** cited by recent open models (SmolLM, OLMo, Ouro, Qwen) and MrCogito experiment plans.

The headline table reports sequence-length estimates from a **1000-row shuffled streaming sample** where a dataset is directly loadable with the current HF/Datasets tooling. Catalog rows with `—` are still useful references, but were gated, format-blocked, or not part of the measured candidate set.

| Artifact | Role |
| --- | --- |
| `analysis/training_dataset_catalog.json` | Machine-readable catalog (models + datasets) |
| `analysis/long_dataset_candidates.json` | HF load configs for sequence-length sampling |
| `playground/long_dataset_seq_len_analysis.ipynb` | Sequence-length charts + interpretation |
| `analysis/dataset_seqlen_distribution.py` | CPU streaming sampler |

Updated: **2026-06-18**

In [1]:
from __future__ import annotations

import json
from pathlib import Path

from IPython.display import Markdown, display

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "playground" else NOTEBOOK_DIR

CATALOG_PATH = REPO_ROOT / "analysis" / "training_dataset_catalog.json"
SEQLEN_SUMMARY_PATH = REPO_ROOT / "Cache" / "Evaluation_reports" / "seqlen_model_mix_1k_shuffle" / "seqlen_dist_summary.json"
CANDIDATES_PATH = REPO_ROOT / "analysis" / "long_dataset_candidates.json"
SAMPLE_LABEL = "1000-row shuffled sample (streaming shuffle buffer=10k, seed=42)"

catalog = json.loads(CATALOG_PATH.read_text())
candidates = json.loads(CANDIDATES_PATH.read_text())
seqlen = json.loads(SEQLEN_SUMMARY_PATH.read_text()) if SEQLEN_SUMMARY_PATH.exists() else []
seqlen_by_name = {row["name"]: row for row in seqlen}

print(f"Catalog: {len(catalog['datasets'])} datasets, {len(catalog['models'])} model programs")
print(f"Measured ({SAMPLE_LABEL}): {len(seqlen)} datasets")

Catalog: 33 datasets, 11 model programs
Measured (1000-row shuffled sample (streaming shuffle buffer=10k, seed=42)): 13 datasets


In [2]:
def md_table(rows: list[dict], columns: list[str]) -> str:
    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join("---" for _ in columns) + " |"
    body = ["| " + " | ".join(str(r.get(c, "")) for c in columns) + " |" for r in rows]
    return "\n".join([header, sep, *body])


def pct(meas: dict | None, threshold: int) -> str:
    if not meas:
        return "—"
    return f"{meas['longer_than_pct'].get(str(threshold), 0):.1f}"


def stat(meas: dict | None, name: str) -> str:
    if not meas:
        return "—"
    return f"{meas['stats'][name]:,}"


dataset_table_rows = []
for ds in catalog["datasets"]:
    meas = seqlen_by_name.get(ds.get("sample_name", ""))
    dataset_table_rows.append({
        "id": ds["id"],
        "hf_id": ds["hf_id"],
        "scale": ds["scale"],
        "type": ds["type"],
        "dataset_rows": ds.get("dataset_rows", "—"),
        "sample_docs": f"{meas['total_docs']:,}" if meas else "—",
        ">512%": pct(meas, 512),
        ">1k%": pct(meas, 1024),
        ">2k%": pct(meas, 2048),
        ">4k%": pct(meas, 4096),
        ">8k%": pct(meas, 8192),
        "median": stat(meas, "p50"),
        "max": stat(meas, "max"),
        "used_by": ", ".join(ds.get("used_by", [])),
    })

columns = [
    "id", "hf_id", "scale", "type", "dataset_rows", "sample_docs",
    ">512%", ">1k%", ">2k%", ">4k%", ">8k%", "median", "max", "used_by",
]

display(Markdown(
    f"## All cataloged datasets ({SAMPLE_LABEL} where measured)\n"
    + md_table(dataset_table_rows, columns)
))


## All cataloged datasets (1000-row shuffled sample (streaming shuffle buffer=10k, seed=42) where measured)
| id | hf_id | scale | type | dataset_rows | sample_docs | >512% | >1k% | >2k% | >4k% | >8k% | median | max | used_by |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| fineweb_edu | HuggingFaceFW/fineweb-edu | ~1.3T full; 10B sample on HF | pretrain | 9,672,101 | 1,000 | 59.6 | 24.9 | 8.6 | 2.2 | 0.6 | 633 | 17,108 | smollm2, smollm3, qwen25, qwen3, mrcogito |
| dclm_baseline | mlfoundations/dclm-baseline-1.0 | ~2.6T tokens | pretrain | ~3B | 1,000 | 61.5 | 36.0 | 16.1 | 6.9 | 2.1 | 711 | 77,807 | smollm3, olmo2, ettin |
| fineweb2 | HuggingFaceFW/fineweb-2 | multilingual web; 100BT+ samples per language | pretrain | — | — | — | — | — | — | — | — | — | smollm3 |
| finepdfs_100BT_pdf_pretrain | HuggingFaceFW/finepdfs_100BT | 100B sample; ~3T parent | pretrain | 29,904,625 | 1,000 | 74.2 | 54.6 | 34.2 | 20.2 | 10.6 | 1,199 | 78,552 | qwen3, mrcogito_e05 |
| finemath | HuggingFaceTB/finemath | tens of B tokens per subset | pretrain | 21,405,610 | 1,000 | 73.3 | 38.1 | 14.7 | 4.5 | 1.4 | 806 | 86,919 | smollm2, smollm3, qwen25, qwen3 |
| stack_edu | HuggingFaceTB/stack-edu | 125B tokens total | code pretrain | 25,286,012 | 1,000 | 44.2 | 21.9 | 8.0 | 2.8 | 1.0 | 430 | 21,727 | smollm2, smollm3, qwen3 |
| megamath | LLM360/MegaMath | 371B tokens (paper); web/code/synthetic variants | pretrain | — | — | — | — | — | — | — | — | — | ouro, smollm3, qwen3 |
| dolmino_mix | allenai/dolmino-mix-1124 | 1.124T tokens | pretrain mix | 3,523,358 | — | — | — | — | — | — | — | — | olmo2 |
| olmo_mix | allenai/olmo-mix-1124 | part of OLMo 2 training | pretrain mix | 87,719 | 1,000 | 60.8 | 31.0 | 12.5 | 4.5 | 1.5 | 659 | 27,081 | olmo2 |
| dolma | allenai/dolma | 2.3T tokens (v1.7) | pretrain | — | — | — | — | — | — | — | — | — | olmo2 |
| pes2o | allenai/peS2o | scientific papers (S2ORC-derived) | pretrain | — | — | — | — | — | — | — | — | — | olmo2, ettin, dolma |
| nemotron_cc | nvidia/Nemotron-CC-v2 | 6.3T tokens | pretrain | — | — | — | — | — | — | — | — | — | ouro |
| map_cc | mlfoundations/MAP-CC | ~800B tokens (Ouro Stage 1) | pretrain | — | — | — | — | — | — | — | — | — | ouro |
| ultra_fineweb_zh | HuggingFaceFW/ultrafineweb-zh | ~120B tokens (Ouro Stage 1) | pretrain | — | — | — | — | — | — | — | — | — | ouro |
| opencoder_pretrain | OpenCoder-LLM/opencoder-pretrain | ~450B tokens (Ouro Stage 1) | code pretrain | — | — | — | — | — | — | — | — | — | ouro |
| opc_annealing | OpenCoder-LLM/opc-annealing-corpus | ~7B tokens | annealing | 3,238,929 | 1,000 | 25.9 | 6.7 | 1.2 | 0.0 | 0.0 | 326 | 4,052 | ouro |
| prolong_64k | princeton-nlp/prolong-data-64K | 20B tokens (Ouro LongCT) | long-context | — | — | — | — | — | — | — | — | — | ouro |
| cosmopedia_v2 | HuggingFaceTB/cosmopedia-v2 | 39M synthetic textbooks | synthetic pretrain | 39M | 1,000 | 89.4 | 18.6 | 0.1 | 0.0 | 0.0 | 741 | 2,058 | smollm |
| longblocks_doc_qa_reasoning | utter-project/LongBlocks | 57.6k rows | SFT / long-context | 193,894 | 1,000 | 99.9 | 96.7 | 85.0 | 55.0 | 45.3 | 5,165 | 198,593 | mrcogito_e05 |
| openmathreasoning_cot | nvidia/OpenMathReasoning | 3.2M CoT + 1.7M TIR | reasoning | 3,201,061 | 1,000 | 100.0 | 99.9 | 94.9 | 79.7 | 47.7 | 7,865 | 23,307 | ouro, smollm3, mrcogito |
| openthoughts3_math_code_science | open-thoughts/OpenThoughts3-1.2M | 1.2M rows | reasoning SFT | 1,200,000 | 1,000 | 100.0 | 100.0 | 99.9 | 99.7 | 97.9 | 19,711 | 24,216 | ouro, smollm3, mrcogito |
| big_reasoning_traces | allenai/big-reasoning-traces | ~2.5B tokens | reasoning mid-train | 676,665 | 1,000 | 99.9 | 95.2 | 78.4 | 51.5 | 28.5 | 4,362 | 29,806 | olmo2, ouro |
| ettin_pretraining | jhu-clsp/ettin-pretraining-data | 1.7T tokens | pretrain mix | — | — | — | — | — | — | — | — | — | ettin |
| dolma3_mix_6t | allenai/dolma3_mix-6T | 6T tokens | pretrain mix | — | — | — | — | — | — | — | — | — | olmo3 |
| dolma3_dolmino_100b | allenai/dolma3_dolmino_mix-100B-1025 | 100B tokens | mid-train / anneal | — | — | — | — | — | — | — | — | — | olmo3 |
| dolma3_longmino_100b | allenai/dolma3_longmino_mix-100B-1125 | 100B tokens | long-context stage | — | — | — | — | — | — | — | — | — | olmo3 |
| fineweb2_hq | epfml/FineWeb2-HQ | subset of FineWeb-2 (20 langs) | pretrain | — | — | — | — | — | — | — | — | — | smollm3 |
| ultra_fineweb | openbmb/Ultra-FineWeb | ~1.8T EN + 120B ZH | pretrain | — | — | — | — | — | — | — | — | — | ouro |
| refinecode | OpenCoder-LLM/refinecode | 960B tokens | code pretrain | — | — | — | — | — | — | — | — | — | ouro, smollm3 |
| starcoderdata | bigcode/starcoderdata | ~250B tokens | code pretrain | — | — | — | — | — | — | — | — | — | smollm2 |
| the_stack_v2 | bigcode/the-stack-v2 | multi-T tokens | code pretrain | — | — | — | — | — | — | — | — | — | smollm3 |
| ettin_extension | jhu-clsp/ettin-extension-data | 250B tokens | mid-train | — | — | — | — | — | — | — | — | — | ettin |
| ettin_decay | jhu-clsp/ettin-decay-data | 50-100B tokens | decay/anneal | — | — | — | — | — | — | — | — | — | ettin |

In [3]:
for model in catalog["models"]:
    rows = []
    for ref in model.get("dataset_refs", []):
        ds = next((d for d in catalog["datasets"] if d["id"] == ref), None)
        if not ds:
            rows.append({"ref": ref, "hf_id": "(not in catalog)", "scale": "", "type": "", "notes": ""})
            continue
        rows.append({
            "ref": ref,
            "hf_id": ds["hf_id"],
            "scale": ds["scale"],
            "type": ds["type"],
            "notes": ds.get("load_notes", ""),
        })
    stages = "; ".join(f"{s['name']}: {s.get('mix', '')}" for s in model.get("stages", []))
    display(Markdown(
        f"### {model['name']} ({model.get('total_tokens', '?')})\n"
        f"Sources: {', '.join(model.get('sources', []))}\n\n"
        + (f"Stages: {stages}\n\n" if stages else "")
        + (f"*{model.get('notes', '')}*\n\n" if model.get('notes') else "")
        + md_table(rows, ["ref", "hf_id", "scale", "type", "notes"])
    ))

### SmolLM3-3B (11.2T pretrain + ~140B mid-train)
Sources: https://huggingface.co/blog/smollm3, https://github.com/huggingface/smollm

Stages: Stage 1 (0-8T): 85% web, 12% code, 3% math; Stage 2 (8-10T): 75% web, 15% code, 10% math; Stage 3 (10-11.1T): 63% web, 24% code, 13% math (+ OpenMathReasoning)

| ref | hf_id | scale | type | notes |
| --- | --- | --- | --- | --- |
| fineweb_edu | HuggingFaceFW/fineweb-edu | ~1.3T full; 10B sample on HF | pretrain | Streaming OK; use length_column=token_count |
| dclm_baseline | mlfoundations/dclm-baseline-1.0 | ~2.6T tokens | pretrain | Streaming OK; tokenize text column |
| fineweb2 | HuggingFaceFW/fineweb-2 | multilingual web; 100BT+ samples per language | pretrain | Pick a language config; no single 'sample-10BT' config |
| stack_edu | HuggingFaceTB/stack-edu | 125B tokens total | code pretrain | Original uses SWH blob_id — needs download_contents. Use meryyllebr543/stack-edu-huggingface for direct text. |
| finemath | HuggingFaceTB/finemath | tens of B tokens per subset | pretrain | token_count column; finemath-4plus is higher quality (Stage 2+) |
| megamath | LLM360/MegaMath | 371B tokens (paper); web/code/synthetic variants | pretrain | HF default config is code shards; web subsets may need separate collection URLs |
| openmathreasoning_cot | nvidia/OpenMathReasoning | 3.2M CoT + 1.7M TIR | reasoning | Use cot split |
| fineweb2_hq | epfml/FineWeb2-HQ | subset of FineWeb-2 (20 langs) | pretrain | Higher precision than full FineWeb-2 |
| refinecode | OpenCoder-LLM/refinecode | 960B tokens | code pretrain | Large; language subsets available |
| starcoderdata | bigcode/starcoderdata | ~250B tokens | code pretrain | Curated StarCoder2 subset |
| the_stack_v2 | bigcode/the-stack-v2 | multi-T tokens | code pretrain | Gated; huge; use language subsets |
| cosmopedia_v2 | HuggingFaceTB/cosmopedia-v2 | 39M synthetic textbooks | synthetic pretrain | token_length column; streaming OK |

### SmolLM2 (135M/360M/1.7B) (~11T (family))
Sources: https://huggingface.co/blog/smollm-v2, https://github.com/huggingface/smollm

Stages: Pretrain: FineWeb-Edu + FineMath + code (Stack-Edu)

| ref | hf_id | scale | type | notes |
| --- | --- | --- | --- | --- |
| fineweb_edu | HuggingFaceFW/fineweb-edu | ~1.3T full; 10B sample on HF | pretrain | Streaming OK; use length_column=token_count |
| finemath | HuggingFaceTB/finemath | tens of B tokens per subset | pretrain | token_count column; finemath-4plus is higher quality (Stage 2+) |
| stack_edu | HuggingFaceTB/stack-edu | 125B tokens total | code pretrain | Original uses SWH blob_id — needs download_contents. Use meryyllebr543/stack-edu-huggingface for direct text. |

### OLMo 2 (~4T (OLMo-2 family))
Sources: https://huggingface.co/datasets/allenai/olmo-mix-1124, https://arxiv.org/abs/2412.02595

Stages: Pretrain: Dolmino mix (DCLM, Dolma sources, StarCoder, peS2o, arXiv, ...)

| ref | hf_id | scale | type | notes |
| --- | --- | --- | --- | --- |
| dolmino_mix | allenai/dolmino-mix-1124 | 1.124T tokens | pretrain mix | Streaming OK; text column |
| olmo_mix | allenai/olmo-mix-1124 | part of OLMo 2 training | pretrain mix | Streaming OK |
| dolma | allenai/dolma | 2.3T tokens (v1.7) | pretrain | Large; many subset configs |
| pes2o | allenai/peS2o | scientific papers (S2ORC-derived) | pretrain | datasets>=3.0 rejects legacy script; needs parquet conversion or old datasets pin |
| big_reasoning_traces | allenai/big-reasoning-traces | ~2.5B tokens | reasoning mid-train | num_tokens column |

### OLMo 3 (6T+ staged (Dolma3))
Sources: https://huggingface.co/datasets/allenai/dolma3_mix-6T, https://arxiv.org/abs/2512.13961

Stages: Stage 1: dolma3_mix-6T (6T); Stage 2: dolma3_dolmino_mix-100B-1025; Long context: dolma3_longmino_mix-100B-1125

| ref | hf_id | scale | type | notes |
| --- | --- | --- | --- | --- |
| dolma3_mix_6t | allenai/dolma3_mix-6T | 6T tokens | pretrain mix | Very large; shard/stream. Primary OLMo3 mix per HF card. |
| dolma3_dolmino_100b | allenai/dolma3_dolmino_mix-100B-1025 | 100B tokens | mid-train / anneal | Smaller than full dolmino; OLMo3-7B reproduction stage-2 |
| dolma3_longmino_100b | allenai/dolma3_longmino_mix-100B-1125 | 100B tokens | long-context stage | Long-context CPT mix; not yet seq-len sampled |
| dclm_baseline | mlfoundations/dclm-baseline-1.0 | ~2.6T tokens | pretrain | Streaming OK; tokenize text column |
| pes2o | allenai/peS2o | scientific papers (S2ORC-derived) | pretrain | datasets>=3.0 rejects legacy script; needs parquet conversion or old datasets pin |
| stack_edu | HuggingFaceTB/stack-edu | 125B tokens total | code pretrain | Original uses SWH blob_id — needs download_contents. Use meryyllebr543/stack-edu-huggingface for direct text. |
| finemath | HuggingFaceTB/finemath | tens of B tokens per subset | pretrain | token_count column; finemath-4plus is higher quality (Stage 2+) |

### Ouro (LoopLM) (7.7T)
Sources: https://arxiv.org/abs/2510.25741, https://ouro-llm.github.io/

Stages: Stage 1 pretrain: 73% Nemotron-CC, 13% MAP-CC, 7.5% OpenCoder, 4% MegaMath-web, 2% Ultra-FineWeb-zh; Stage 2 CT anneal (16k seq): Nemotron-CC-HQ, MegaMath-HQ, Nemotron-CC-Math, OpenCoder-Annealing, Nemotron code/SFT; Stage 3 LongCT: ProLong-64K (20B tokens); Stage 4 mid-train + SFT: OpenThoughts3, AceReason, OpenCodeReasoning, Nemotron post-train

| ref | hf_id | scale | type | notes |
| --- | --- | --- | --- | --- |
| nemotron_cc | nvidia/Nemotron-CC-v2 | 6.3T tokens | pretrain | Gated — requires HF auth + license acceptance |
| map_cc | mlfoundations/MAP-CC | ~800B tokens (Ouro Stage 1) | pretrain | HF ID unverified / may not be public standalone |
| ultra_fineweb_zh | HuggingFaceFW/ultrafineweb-zh | ~120B tokens (Ouro Stage 1) | pretrain | HF ID not found at probe time |
| opencoder_pretrain | OpenCoder-LLM/opencoder-pretrain | ~450B tokens (Ouro Stage 1) | code pretrain | Check OpenCoder org for exact subset names |
| megamath | LLM360/MegaMath | 371B tokens (paper); web/code/synthetic variants | pretrain | HF default config is code shards; web subsets may need separate collection URLs |
| opc_annealing | OpenCoder-LLM/opc-annealing-corpus | ~7B tokens | annealing | Multiple configs; streaming OK |
| prolong_64k | princeton-nlp/prolong-data-64K | 20B tokens (Ouro LongCT) | long-context | Mosaic streaming shards — not plain HF text rows; needs mosaic-streaming reader |
| openmathreasoning_cot | nvidia/OpenMathReasoning | 3.2M CoT + 1.7M TIR | reasoning | Use cot split |
| openthoughts3_math_code_science | open-thoughts/OpenThoughts3-1.2M | 1.2M rows | reasoning SFT | conversations column |
| big_reasoning_traces | allenai/big-reasoning-traces | ~2.5B tokens | reasoning mid-train | num_tokens column |
| refinecode | OpenCoder-LLM/refinecode | 960B tokens | code pretrain | Large; language subsets available |
| ultra_fineweb | openbmb/Ultra-FineWeb | ~1.8T EN + 120B ZH | pretrain | EN and ZH configs; Apache-2.0 |

...

### Qwen2.5 (~18T)
Sources: https://arxiv.org/pdf/2412.15115

Stages: Pretrain: web + math + code + multilingual; heavy filtering

*Paper cites categories not exact HF IDs; proxies listed.*

| ref | hf_id | scale | type | notes |
| --- | --- | --- | --- | --- |
| fineweb_edu | HuggingFaceFW/fineweb-edu | ~1.3T full; 10B sample on HF | pretrain | Streaming OK; use length_column=token_count |
| finemath | HuggingFaceTB/finemath | tens of B tokens per subset | pretrain | token_count column; finemath-4plus is higher quality (Stage 2+) |
| stack_edu | HuggingFaceTB/stack-edu | 125B tokens total | code pretrain | Original uses SWH blob_id — needs download_contents. Use meryyllebr543/stack-edu-huggingface for direct text. |
| megamath | LLM360/MegaMath | 371B tokens (paper); web/code/synthetic variants | pretrain | HF default config is code shards; web subsets may need separate collection URLs |

### Kimi / Moonshot (k1.5) (undisclosed)
Sources: https://arxiv.org/abs/2501.12599, https://moonshot.cn/

*k1.5: proprietary pretrain (EN/ZH/code/math/knowledge + multimodal). Partial open refs: BigCode/StarCoder2 pipeline, FineWeb-style scoring, LAION/DataComp/OBELICS for vision. Long context to 131k. No HF pretrain IDs.*

| ref | hf_id | scale | type | notes |
| --- | --- | --- | --- | --- |
| fineweb_edu | HuggingFaceFW/fineweb-edu | ~1.3T full; 10B sample on HF | pretrain | Streaming OK; use length_column=token_count |
| starcoderdata | bigcode/starcoderdata | ~250B tokens | code pretrain | Curated StarCoder2 subset |
| the_stack_v2 | bigcode/the-stack-v2 | multi-T tokens | code pretrain | Gated; huge; use language subsets |
| megamath | LLM360/MegaMath | 371B tokens (paper); web/code/synthetic variants | pretrain | HF default config is code shards; web subsets may need separate collection URLs |

### Ettin (1B/8B family) (1.7T)
Sources: https://huggingface.co/datasets/jhu-clsp/ettin-pretraining-data

Stages: Pretrain: DCLM, CC, StarCoder, Reddit, peS2o, arXiv, StackExchange, Tulu/FLAN, OpenWebMath, Wikipedia

| ref | hf_id | scale | type | notes |
| --- | --- | --- | --- | --- |
| ettin_pretraining | jhu-clsp/ettin-pretraining-data | 1.7T tokens | pretrain mix | MDS format; HF streaming fails without mosaic |
| dclm_baseline | mlfoundations/dclm-baseline-1.0 | ~2.6T tokens | pretrain | Streaming OK; tokenize text column |
| pes2o | allenai/peS2o | scientific papers (S2ORC-derived) | pretrain | datasets>=3.0 rejects legacy script; needs parquet conversion or old datasets pin |
| stack_edu | HuggingFaceTB/stack-edu | 125B tokens total | code pretrain | Original uses SWH blob_id — needs download_contents. Use meryyllebr543/stack-edu-huggingface for direct text. |

### MrCogito E01/E02/E03 (current) (10B sample)
Sources: docs/experiments_specs/E02_*.md

Stages: Reconstruction pretrain: FineWeb-Edu sample-10BT, max_seq_length=512

| ref | hf_id | scale | type | notes |
| --- | --- | --- | --- | --- |
| fineweb_edu | HuggingFaceFW/fineweb-edu | ~1.3T full; 10B sample on HF | pretrain | Streaming OK; use length_column=token_count |

### MrCogito E05 (planned long context) (staged)
Sources: docs/experiments_specs/E05_windowed_decoder_concept_memory.md

Stages: Stage A: FinePDFs-100BT pretrain; Stress tests: LongBlocks, OpenThoughts3

| ref | hf_id | scale | type | notes |
| --- | --- | --- | --- | --- |
| finepdfs_100BT_pdf_pretrain | HuggingFaceFW/finepdfs_100BT | 100B sample; ~3T parent | pretrain | Parquet; token_count column available |
| longblocks_doc_qa_reasoning | utter-project/LongBlocks | 57.6k rows | SFT / long-context | Concatenate document+QA columns |
| openthoughts3_math_code_science | open-thoughts/OpenThoughts3-1.2M | 1.2M rows | reasoning SFT | conversations column |

In [24]:
detail_rows = []
for ds in catalog["datasets"]:
    detail_rows.append({
        "id": ds["id"],
        "domains": ds.get("domains", ""),
        "typical_length": ds.get("typical_length", ""),
        "variety": ds.get("variety", ""),
        "load_notes": ds.get("load_notes", ""),
    })

display(Markdown("## Dataset detail (length + variety + loading)\n" + md_table(detail_rows, ["id", "domains", "typical_length", "variety", "load_notes"])))

## Dataset detail (length + variety + loading)
| id | domains | typical_length | variety | load_notes |
| --- | --- | --- | --- | --- |
| fineweb_edu | web, educational filter | median ~631 tokens (HF stats); short-doc heavy | broad web knowledge, English-primary | Streaming OK; use length_column=token_count |
| dclm_baseline | web (DCLM pipeline) | web-crawl; similar short-doc tail to FineWeb | high-quality open web crawl | Streaming OK; tokenize text column |
| fineweb2 | multilingual web | web-crawl; language-specific | 1000+ languages; SmolLM3 uses ~12% multilingual | Pick a language config; no single 'sample-10BT' config |
| finepdfs_100BT_pdf_pretrain | PDF documents | p50 ~1.5k; long tail to 50k+ | varied PDF-derived docs; better long tail than FineWeb-Edu | Parquet; token_count column available |
| finemath | math web | math pages; moderate length | math-only but high quality | token_count column; finemath-4plus is higher quality (Stage 2+) |
| stack_edu | educational code from The Stack v2 | file-level; variable, often 500-8k | 15 languages; educational filter | Original uses SWH blob_id — needs download_contents. Use meryyllebr543/stack-edu-huggingface for direct text. |
| megamath | math web, math code, synthetic QA | web avg ~2.3k tokens; synthetic shorter | math-focused; used by Ouro and SmolLM3 Stage 2+ | HF default config is code shards; web subsets may need separate collection URLs |
| dolmino_mix | DCLM, Dolma, StarCoder, peS2o, arXiv, StackExchange, ... | mixed; web-dominated short docs | very high — OLMo 2 canonical mix | Streaming OK; text column |
| olmo_mix | same family as Dolmino | mixed web | OLMo 2 release mix | Streaming OK |
| dolma | CC, RefinedWeb, StarCoder, Reddit, peS2o, arXiv, ... | mixed | foundational open corpus | Large; many subset configs |
| pes2o | STEM papers | 5k-30k tokens typical for full papers | scientific; coherent long docs | datasets>=3.0 rejects legacy script; needs parquet conversion or old datasets pin |
| nemotron_cc | refined Common Crawl | web; Ouro uses 16k seq in anneal — implies moderate+ | high-quality web; Ouro primary corpus | Gated — requires HF auth + license acceptance |
| map_cc | web CC variant | web | multilingual web | HF ID unverified / may not be public standalone |
| ultra_fineweb_zh | Chinese web | web | Chinese only; Ouro dropped after Stage 1 | HF ID not found at probe time |
| opencoder_pretrain | code | file/repo level | code | Check OpenCoder org for exact subset names |
| opc_annealing | synthetic code QA | code snippets; moderate | code-focused synthetic | Multiple configs; streaming OK |
| prolong_64k | books, arXiv, code — packed to 64k | pre-packed 65,536 tokens | long coherent docs | Mosaic streaming shards — not plain HF text rows; needs mosaic-streaming reader |
| cosmopedia_v2 | textbooks, stories, knowledge | token_length column available | synthetic educational; SmolLM family | token_length column; streaming OK |
| longblocks_doc_qa_reasoning | books, arXiv, Wiki, Stack + synthetic QA | p50 ~5k; 33% >32k in 100-row sample | best varied ultra-long SFT candidate | Concatenate document+QA columns |
| openmathreasoning_cot | math CoT | p50 ~7k tokens | math reasoning traces | Use cot split |
| openthoughts3_math_code_science | math, code, science | p50 ~20k; consistently long | reasoning traces; not broad world knowledge | conversations column |
| big_reasoning_traces | compiled reasoning traces | p50 ~1.4k; weaker ultra-long tail | OLMo 2 reasoning mix component | num_tokens column |
| ettin_pretraining | DCLM, code, peS2o, arXiv, StackExchange, ... | mixed | excellent blend; MDS/TAR format | MDS format; HF streaming fails without mosaic |
| dolma3_mix_6t | OLMo3 stage-1 successor to olmo-mix | mixed | newest fully open mixed pretrain (OLMo3 lineage) | Very large; shard/stream. Primary OLMo3 mix per HF card. |
| dolma3_dolmino_100b | HQ subset mix | mixed | OLMo3 stage-2 annealing-quality mix | Smaller than full dolmino; OLMo3-7B reproduction stage-2 |
| dolma3_longmino_100b | long documents | long-doc emphasis | OLMo3 explicit long-context stage; high priority for E05-style work | Long-context CPT mix; not yet seq-len sampled |
| fineweb2_hq | multilingual web, model-filtered HQ | web-crawl | SmolLM3 uses alongside FineWeb-2 | Higher precision than full FineWeb-2 |
| ultra_fineweb | web EN+ZH | web | Ouro cites Ultra-FineWeb-zh; openbmb hosts EN+ZH | EN and ZH configs; Apache-2.0 |
| refinecode | 607 languages, source files | file-level; can be long | OpenCoder core; Ouro stage-1 code component | Large; language subsets available |
| starcoderdata | code files | file-level | SmolLM2 stage-1 code; easier than full Stack v2 | Curated StarCoder2 subset |
| the_stack_v2 | 600+ langs | file/repo level | SmolLM3 12-24% code mix | Gated; huge; use language subsets |
| ettin_extension | HQ filtered mix | mixed | Ettin stage-2; Dolmino-style | MDS format (MosaicML) |
| ettin_decay | premium sources | mixed | Ettin final decay phase | MDS format |

## Model-specific notes

### Ouro (LoopLM, arxiv:2510.25741)
7.7T tokens, all open-source. Stage 1: Nemotron-CC (6.3T) + MAP-CC + OpenCoder + MegaMath-web. Stage 2 CT anneal at **16k seq**. Stage 3: **ProLong-64K** (20B). Stage 4 mid-train: OpenThoughts3, AceReason, OpenCodeReasoning.

### SmolLM3 (HuggingFaceTB blog)
11.2T three-stage mix evolving from 85/12/3 (web/code/math) to 63/24/13. Key public HF IDs: FineWeb-Edu, DCLM, FineWeb-2 (multilingual), Stack-Edu, FineMath, MegaMath, OpenMathReasoning.

### OLMo 2 (allenai)
Dolmino-mix-1124 (1.124T) and olmo-mix-1124. Components include DCLM, Dolma sources, StarCoder, peS2o, arXiv, StackExchange.

### Qwen3 (arxiv:2505.09388)
~36T tokens, 119 languages. S1 general >30T, S2 +5T STEM/code/reasoning, S3 long-context (75% samples 16k–32k). Most corpus is proprietary; public HF proxies listed in catalog.

### Kimi / Moonshot
No public training corpus release as of 2026-06.

## Rerun sequence-length sampling

```bash
uv run python analysis/dataset_seqlen_distribution.py \
  --candidates analysis/long_dataset_candidates.json \
  --tokenizer HuggingFaceTB/SmolLM2-135M \
  --max_docs 1000 \
  --shuffle \
  --shuffle_buffer_size 10000 \
  --seed 42 \
  --out_dir Cache/Evaluation_reports/seqlen_model_mix_1k_shuffle
```

Then open `playground/training_dataset_catalog.ipynb` for the catalog table or `playground/long_dataset_seq_len_analysis.ipynb` for charts.